# Модуль 14. Разбор кейсов: Передаваемый лабораторный конвейер

**Модуль 14. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** molecules.csv; дополнительный ошибочный CSV без нужного столбца.

Проверить работу с чужим CSV, сохранность строк и один вычислительный контракт для CLI и интерфейса.

**Workflow:** CSV contract → ядро → журнал всех строк → паспорт → CLI → проверка передачи.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab14_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%2014_SDLS-workflow%20development_web%20app.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab14/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Контракт вычислительного ядра

Приложение начинается с проверяемой функции. В notebook разбираем готовый process_csv: обязательные колонки, уникальные ID, все статусы и паспорт политики. Интерфейс не должен подменять химический контракт. Статус eligible ограничен правилами вычислительного отбора.

**Перед запуском:** Вызовите табличную функцию напрямую и проверьте все статусы.

In [2]:
from pipeline import process_csv
data=(DATA/'molecules.csv').read_bytes()
rows,passport=process_csv(data)
display(pd.DataFrame(rows)[['id','raw_smiles','parent_smiles','status','reason']])
assert len(rows)==12 and sum(r['status']=='invalid' for r in rows)==2


[14:01:22] Initializing MetalDisconnector
[14:01:22] Running MetalDisconnector
[14:01:22] Initializing Normalizer
[14:01:22] Running Normalizer
[14:01:22] Initializing MetalDisconnector
[14:01:22] Running MetalDisconnector
[14:01:22] Initializing Normalizer
[14:01:22] Running Normalizer
[14:01:22] Running LargestFragmentChooser
[14:01:22] Running Uncharger
[14:01:22] Initializing MetalDisconnector
[14:01:22] Running MetalDisconnector
[14:01:22] Initializing Normalizer
[14:01:22] Running Normalizer
[14:01:22] Initializing MetalDisconnector
[14:01:22] Running MetalDisconnector
[14:01:22] Initializing Normalizer
[14:01:22] Running Normalizer
[14:01:22] Running LargestFragmentChooser
[14:01:22] Running Uncharger
[14:01:22] Initializing MetalDisconnector
[14:01:22] Running MetalDisconnector
[14:01:22] Initializing Normalizer
[14:01:22] Running Normalizer
[14:01:22] Initializing MetalDisconnector
[14:01:22] Running MetalDisconnector
[14:01:22] Initializing Normalizer
[14:01:22] Running Norma

,id,raw_smiles,parent_smiles,status,reason
0,M01,CCO,CCO,eligible,No listed alert; activity remains unknown
1,M02,OCC,CCO,eligible,No listed alert; activity remains unknown
2,M03,CC(=O)[O-].[Na+],CC(=O)O,eligible,No listed alert; activity remains unknown
3,M04,CC(=O)O,CC(=O)O,eligible,No listed alert; activity remains unknown
4,M05,Clc1ccccc1,Clc1ccccc1,eligible,No listed alert; activity remains unknown
5,M06,Oc1ccccc1O,Oc1ccccc1O,review,PAINS:catechol_A(92)
6,M07,c1ccccc1,c1ccccc1,eligible,No listed alert; activity remains unknown
7,M08,not_a_smiles,NaN,invalid,Invalid SMILES
8,M09,,NaN,invalid,SMILES must be a nonempty string
9,M10,C[C@H](O)C(=O)O,C[C@H](O)C(=O)O,eligible,No listed alert; activity remains unknown


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Контракт вычислительного ядра — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Неправильный CSV как готовый кейс

Испытываем отсутствующий smiles и повторный ID. Это ожидаемый контролируемый отказ, а не падение интерфейса с непонятным traceback. Ошибка схемы отличается от invalid SMILES внутри корректной таблицы: во втором случае строка сохраняется в отчёте.

**Перед запуском:** Запустите тот же конвейер как отдельный CLI-процесс.

In [3]:
negative_inputs=[b'id,wrong\nX,CCO\n',b'id,smiles\nX,CCO\nX,CCN\n']
errors=[]
for invalid_csv in negative_inputs:
    try:
        process_csv(invalid_csv)
    except ValueError as error:
        errors.append(str(error))
    else:
        raise AssertionError('Нарушение CSV контракта принято')
print(errors)
assert len(errors)==2
assert any('unique' in error for error in errors)


['CSV requires id and smiles columns', 'IDs must be nonempty and unique']


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Неправильный CSV как готовый кейс — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Переходим в IDE

Разработку CLI и Streamlit ведём в файлах проекта через VS Code WSL или другую IDE. Notebook остаётся журналом разбора и результатов. Ядро, CLI и GUI должны использовать одну вычислительную политику и выдавать воспроизводимый паспорт. Готовый CSV pipeline ещё не является полным научным досье или доказательством самостоятельного переноса.

**Перед запуском:** Сравните ID и hash отчётов, затем откройте локальный Streamlit при желании.

In [4]:
assert passport['input_sha256']==hashlib.sha256(data).hexdigest()
assert len({r['id'] for r in rows})==len(rows)
print(passport)
artifact={'rows':len(rows),'passport':passport,'negative_input_errors':errors,
          'scope':'reference core only; application development and tests take place in IDE'}


{'input_sha256': 'f1904ebd8af5789d383f9593365bf5443abf7aa60106db0ecfb50d4fff628465', 'python': '3.12.3', 'rdkit': '2026.09.1', 'rows': 12, 'policy': 'teaching-parent-v1;neutralize;tautomer=False'}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Переходим в IDE — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

CLI и прямой вызов сохраняют 12 строк и hash; отсутствующий столбец/повторный ID отклоняются; Streamlit — тот же pipeline.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':14,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab14/worked/benchmark.json


## 5. Разработка приложения — в IDE

Откройте корень проекта в VS Code WSL. Создайте `student_work/module14/cli.py` и `app.py`; отладчик, терминал и тесты запускайте там. Полная [инструкция IDE](../ide/README.md). Здесь сервер не запускается. Во втором notebook запишите команды, результаты и ссылки на исходники.

## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab14_practice.ipynb](lab14_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.